# 模拟题 A — Feature Selection: Hotel Review Data

**关于数据集：** `hotel_reviews.csv`

该数据集包含 5000 条酒店住客评价记录，每行代表一次入住体验。

**变量说明：**
- `guest_id`: 住客 ID（需要删除）
- `nationality`: 住客国籍（需要删除）
- `loyalty_member`: 是否为忠诚度会员 Yes/No（需要删除）
- `stay_duration`: 入住天数
- `room_rate`: 每晚房价（美元）
- `checkin_delay`: 入住等待时间（分钟）
- `checkout_delay`: 退房等待时间（分钟）
- `room_cleanliness`: 房间清洁评分 (1–5)
- `staff_service`: 员工服务评分 (1–5)
- `food_quality`: 餐饮质量评分 (1–5)
- `amenities`: 设施评分 (1–5)
- `location_score`: 地理位置评分 (1–5)
- `value_for_money`: 性价比评分 (1–5)
- `noise_level`: 噪音评分 (1–5，5=非常安静)
- `overall_rating`: 整体满意度评分 (1–5)【目标变量】

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import math

In [ ]:
# A1 ──────────────────────────────────────────────────────────────────
# 读入 hotel_reviews.csv。删除 guest_id, nationality, loyalty_member。
# 使用 train_test_split 进行分层抽样（strata=overall_rating），
# 70% 训练集，random_state=2024。
# train 样本中 stay_duration 的中位数是多少？

# data = pd.read_csv('hotel_reviews.csv')
# data2 = data.drop(columns=['guest_id', 'nationality', 'loyalty_member'])
# y = data2['overall_rating']
# X = data2.drop('overall_rating', axis=1)
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, train_size=0.7, random_state=2024, stratify=y)
# print(X_train['stay_duration'].median())

In [ ]:
# A2 ──────────────────────────────────────────────────────────────────
# 计算各预测变量与 overall_rating 的双变量相关系数（绝对值）。
# 哪个变量与 overall_rating 的相关性最弱？

# train_full = X_train.copy()
# train_full['overall_rating'] = y_train.values
# corr_sat = train_full.corr()['overall_rating'].drop('overall_rating').abs()
# print(corr_sat.sort_values())

In [ ]:
# A3 ──────────────────────────────────────────────────────────────────
# 计算预测变量之间的相关矩阵。哪对变量的相关性最强？

# pred_corr = X_train.corr().abs()
# pairs = []
# cols = X_train.columns
# for i in range(len(cols)):
#     for j in range(i+1, len(cols)):
#         pairs.append((cols[i], cols[j], pred_corr.loc[cols[i], cols[j]]))
# pairs_df = pd.DataFrame(pairs, columns=['var1','var2','corr']).sort_values('corr', ascending=False)
# print(pairs_df.head(5))

In [ ]:
# A4 ──────────────────────────────────────────────────────────────────
# 用 statsmodels 对 train 数据进行全变量线性回归（OLS）。
# 哪些变量在 p < 0.05 水平上显著？

# X_train_sm = sm.add_constant(X_train)
# ols_model = sm.OLS(y_train, X_train_sm).fit()
# print(ols_model.summary())
# sig = ols_model.pvalues[ols_model.pvalues < 0.05].drop('const', errors='ignore')
# print('显著变量:', sig.index.tolist())

In [ ]:
# A5 ──────────────────────────────────────────────────────────────────
# 计算各预测变量的 VIF。哪些变量的 VIF > 5？

# vif_data = pd.DataFrame()
# vif_data['feature'] = X_train.columns
# vif_data['VIF'] = [variance_inflation_factor(X_train.values.astype(float), i)
#                    for i in range(X_train.shape[1])]
# print(vif_data.sort_values('VIF', ascending=False).round(3))
# print('VIF > 5:', vif_data[vif_data['VIF'] > 5]['feature'].tolist())

In [ ]:
# A6 ──────────────────────────────────────────────────────────────────
# 使用 Forward Variable Selection（SequentialFeatureSelector,
# n_features_to_select='auto', scoring='r2', cv=5）。
# 哪些变量被选中？

# lr = LinearRegression()
# sfs_fwd = SequentialFeatureSelector(lr, n_features_to_select='auto',
#                                      direction='forward', scoring='r2', cv=5)
# sfs_fwd.fit(X_train, y_train)
# print('Forward 选中:', X_train.columns[sfs_fwd.get_support()].tolist())

In [ ]:
# A7 ──────────────────────────────────────────────────────────────────
# 使用 Backward Variable Selection（direction='backward'）。
# 哪些变量被选中？与 Forward 结果是否相同？

# sfs_bwd = SequentialFeatureSelector(lr, n_features_to_select='auto',
#                                      direction='backward', scoring='r2', cv=5)
# sfs_bwd.fit(X_train, y_train)
# print('Backward 选中:', X_train.columns[sfs_bwd.get_support()].tolist())

In [ ]:
# A8 ──────────────────────────────────────────────────────────────────
# 用 StandardScaler 标准化特征后，使用 LassoCV（cv=5）进行特征选择。
# 哪些变量的系数被缩减为 0？

# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled  = scaler.transform(X_test)
# lasso_cv = LassoCV(cv=5, random_state=2024)
# lasso_cv.fit(X_train_scaled, y_train)
# coef = pd.Series(lasso_cv.coef_, index=X_train.columns)
# print(coef.round(4))
# print('缩减为 0:', coef[coef == 0].index.tolist())

In [ ]:
# A9 ──────────────────────────────────────────────────────────────────
# 仅使用 Lasso 选中的变量拟合线性回归。test 样本的 R² 是多少？

# lasso_selected = coef[coef != 0].index.tolist()
# lr_lasso = LinearRegression()
# lr_lasso.fit(X_train[lasso_selected], y_train)
# r2_test = r2_score(y_test, lr_lasso.predict(X_test[lasso_selected]))
# print(f'Lasso LR Test R²: {round(r2_test, 4)}')

In [ ]:
# A10 ─────────────────────────────────────────────────────────────────
# 使用 PCA 进行降维，保留约 65% (±2%) 的方差。
# 需要保留几个主成分？打印每个主成分数量对应的累积方差。

# pca_full = PCA()
# pca_full.fit(X_train_scaled)
# cumvar = np.cumsum(pca_full.explained_variance_ratio_)
# for i, v in enumerate(cumvar):
#     print(f'{i+1} components: {round(v*100,2)}%')

In [ ]:
# A11 ─────────────────────────────────────────────────────────────────
# 用 A10 保留的主成分预测 overall_rating（线性回归）。
# Train R² 和 Test R² 分别是多少？
# 与 Lasso 线性回归相比，哪个方法效果更好？为什么？

# n_comp = ???  # 根据 A10 填入
# pca = PCA(n_components=n_comp)
# X_train_pca = pca.fit_transform(X_train_scaled)
# X_test_pca  = pca.transform(X_test_scaled)
# lr_pca = LinearRegression()
# lr_pca.fit(X_train_pca, y_train)
# print('PCA Train R²:', round(r2_score(y_train, lr_pca.predict(X_train_pca)), 4))
# print('PCA Test  R²:', round(r2_score(y_test,  lr_pca.predict(X_test_pca)),  4))

# 📌 概念题：
# PCA 降维的优点是什么？为什么 PCA 的 R² 可能低于直接用原始变量的模型？
# 提示：PCA 捕捉方差最大的方向，不一定是与 y 最相关的方向。

In [ ]:
# A12 ─────────────────────────────────────────────────────────────────
# 【概念题】比较以下四种特征选择/降维方法，分别说明适用场景：
# (1) 双变量相关系数筛选
# (2) 全变量 OLS（统计显著性）
# (3) Sequential Feature Selection（Forward/Backward）
# (4) Lasso
# (5) PCA

# 答案要点：
# (1) 相关系数：简单快速，只看一对一关系，忽略多变量联合效应 → 适合初步筛查
# (2) OLS显著性：考虑所有变量同时存在时的效应 → 适合推断（Inference）
# (3) SFS：自动搜索最优子集，计算量大 → 适合中等变量数
# (4) Lasso：正则化，自动将不重要变量系数缩减为0 → 适合高维数据
# (5) PCA：降维不选变量，生成不相关的主成分 → 适合多重共线性严重或高维场景